# LeetCode 488 — Zuma Game bằng Python

- Link: https://leetcode.com/problems/zuma-game/
- Độ khó: **Hard**
- Chủ đề: Tìm kiếm trạng thái (BFS / DFS + memoization), Stack, Cắt tỉa (pruning)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def findMinStep(self, board: str, hand: str) -> int:

```

## 1. Mục tiêu học

1. Mô hình hóa một trò chơi thành **không gian trạng thái**: trạng thái = (dãy bóng trên bàn, bóng còn trong tay).
2. Viết hàm **"dọn bóng"** (xóa nhóm ≥ 3 bóng cùng màu, xóa dây chuyền) bằng **stack** — một kỹ năng hay gặp (giống LC 1209).
3. Dùng **BFS** để tìm số bước ít nhất và **DFS + memo** (`functools.lru_cache`) để tránh tính lại trạng thái trùng.
4. Hiểu và kiểm chứng các **quy tắc cắt tỉa** (chỉ chèn bóng ở vị trí "có ích"), kể cả trường hợp bẫy `"RRWWRRBBRR", "WB"`.
5. Biết cách đối chiếu lời giải có cắt tỉa với brute force không cắt tỉa trên dữ liệu ngẫu nhiên.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Trên bàn có một hàng bóng màu, mô tả bằng chuỗi `board` (mỗi ký tự là một màu trong `R` đỏ, `Y` vàng, `B` xanh dương, `G` xanh lá, `W` trắng). Trong tay bạn có các bóng, mô tả bằng chuỗi `hand`.

Mỗi lượt:
1. Chọn **một** bóng trong tay, chèn vào **bất kỳ vị trí nào** của hàng (giữa hai bóng, hoặc ở hai đầu).
2. Nếu có một nhóm **từ 3 bóng liên tiếp cùng màu trở lên**, nhóm đó bị xóa. Sau khi xóa, hai phần hai bên dính lại; nếu lại tạo nhóm ≥ 3 thì tiếp tục xóa (**phản ứng dây chuyền**), cho tới khi không còn nhóm nào.
3. Nếu bàn trống → thắng.

Trả về **số bóng ít nhất** cần chèn để dọn sạch bàn; nếu không thể thì trả về `-1`.

**Ví dụ 1**

```text
Input:  board = "WRRBBW", hand = "RB"
Output: -1
Giải thích: chèn R: WRR[R]BBW -> WBBW; chèn B: WBB[B]W -> WW. Còn WW, hết bóng -> thua.
```

**Ví dụ 2**

```text
Input:  board = "WWRRBBWW", hand = "WRBRW"
Output: 2
Giải thích: WWRR[R]BBWW -> WWBBWW; WWBB[B]WW -> WWWW -> trống.
```

**Ví dụ 3**

```text
Input:  board = "G", hand = "GGGGG"
Output: 2
Giải thích: G[G] -> GG; GG[G] -> trống.
```

**Ràng buộc**

- `1 <= board.length <= 16`
- `1 <= hand.length <= 5`
- `board` và `hand` chỉ gồm `'R','Y','B','G','W'`.
- Trạng thái ban đầu của `board` **không** có nhóm ≥ 3 bóng liên tiếp cùng màu.

In [1]:
# Chạy thử bằng tay ví dụ 2: chèn từng bóng bằng slicing
board = "WWRRBBWW"
b1 = board[:4] + "R" + board[4:]
print("sau chèn R:", b1, "-> xóa RRR ->", b1.replace("RRR", ""))
b2 = "WWBBWW"
b3 = b2[:4] + "B" + b2[4:]
print("sau chèn B:", b3, "-> xóa BBB ->", b3.replace("BBB", ""), "-> xóa WWWW -> rỗng")

sau chèn R: WWRRRBBWW -> xóa RRR -> WWBBWW
sau chèn B: WWBBBWW -> xóa BBB -> WWWW -> xóa WWWW -> rỗng


## 3. Kiến thức nền

### 3.1. Chèn ký tự vào chuỗi bằng slicing

Chuỗi trong Python **bất biến** (không sửa tại chỗ được). Muốn chèn `c` vào vị trí `i` (trước ký tự `s[i]`), ta tạo chuỗi mới: `s[:i] + c + s[i:]`. Các vị trí hợp lệ: `i = 0, 1, …, len(s)` — tức có `len(s) + 1` chỗ chèn.

In [2]:
s = "RGB"
for i in range(len(s) + 1):
    print(i, s[:i] + "W" + s[i:])

0 WRGB
1 RWGB
2 RGWB
3 RGBW


### 3.2. Stack để xóa nhóm ≥ 3 và phản ứng dây chuyền

Ý tưởng: duyệt từng bóng, giữ một stack các cặp `[màu, số lượng liên tiếp]`.
- Khi gặp bóng **khác màu** với đỉnh stack, nếu nhóm ở đỉnh đã có `>= 3` thì **xóa** nhóm đó (pop) — lúc này nhóm bên dưới lộ ra và có thể cùng màu với bóng đang xét → dây chuyền.
- Nếu bóng cùng màu với đỉnh: tăng đếm; ngược lại push nhóm mới.
- Cuối cùng kiểm tra đỉnh một lần nữa.

Tại sao chỉ xóa khi gặp bóng **khác màu** (hoặc hết chuỗi)? Vì nhóm còn có thể dài thêm, ví dụ `RRRR` cũng bị xóa cả 4.

In [3]:
def clean(s):
    stack = []                      # mỗi phần tử: [màu, số lượng]
    for c in s:
        if stack and stack[-1][0] != c and stack[-1][1] >= 3:
            stack.pop()             # xóa nhóm >= 3 vừa kết thúc
        if stack and stack[-1][0] == c:
            stack[-1][1] += 1
        else:
            stack.append([c, 1])
    if stack and stack[-1][1] >= 3:
        stack.pop()
    return "".join(c * n for c, n in stack)

print(repr(clean("WWRRRBBWW")))     # WWBBWW
print(repr(clean("WWBBBWW")))       # dây chuyền -> ""
print(repr(clean("RRWWWRRB")))      # RRRRB? -> xóa WWW, RR+RR = RRRR -> B
print(repr(clean("YYRRRRYB")))      # YYYB -> B
assert clean("WWBBBWW") == "" and clean("RRWWWRRB") == "B"

'WWBBWW'
''
'B'
'B'


### 3.3. Đếm và chuẩn hóa bóng trong tay

Thứ tự bóng trong tay **không quan trọng**: `"RB"` và `"BR"` là cùng một trạng thái. Ta chuẩn hóa bằng cách sắp xếp: `"".join(sorted(hand))`. Điều này giúp memo/visited nhận ra hai trạng thái giống nhau. Bỏ một bóng ở vị trí `j`: `hand[:j] + hand[j+1:]`.

In [4]:
hand = "WRBRW"
h = "".join(sorted(hand))
print("chuẩn hóa:", h)
for j in range(len(h)):
    if j > 0 and h[j] == h[j - 1]:
        continue                     # bỏ qua màu trùng để không thử 2 lần
    print("dùng", h[j], "-> còn", h[:j] + h[j + 1:])

chuẩn hóa: BRRWW
dùng B -> còn RRWW
dùng R -> còn BRWW
dùng W -> còn BRRW


### 3.4. BFS với `collections.deque` và tập `visited`

**BFS (Breadth-First Search — tìm kiếm theo chiều rộng)** duyệt trạng thái theo từng "lớp": lớp 0 là trạng thái ban đầu, lớp 1 là mọi trạng thái sau 1 lượt, ... Trạng thái đích gặp đầu tiên nằm ở lớp nhỏ nhất → **số bước ít nhất**.

- `deque`: hàng đợi hai đầu, `append` và `popleft` đều `O(1)`.
- `visited`: tập các trạng thái đã đưa vào hàng đợi, để không xử lý lặp.

In [5]:
from collections import deque
q = deque([("A", 0)])
q.append(("B", 1)); q.append(("C", 1))
print(q.popleft(), q.popleft(), list(q))
visited = {("WRRBBW", "BR")}
print(("WRRBBW", "BR") in visited)

('A', 0) ('B', 1) [('C', 1)]
True


### 3.5. DFS + memo với `functools.lru_cache`

`@lru_cache(maxsize=None)` ghi nhớ kết quả của hàm theo tham số. Gọi lại với cùng tham số trả ngay kết quả cũ. Tham số phải **hashable** (chuỗi, số, tuple — không phải list).

In [6]:
from functools import lru_cache
calls = 0
@lru_cache(maxsize=None)
def fib(n):
    global calls
    calls += 1
    return n if n < 2 else fib(n - 1) + fib(n - 2)
print(fib(40), "số lần thật sự tính:", calls)

102334155 số lần thật sự tính: 41


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 3: `board = "G"`, `hand = "GGGGG"`.

- Lượt 1: chèn G → `GG` (chưa đủ 3, không xóa).
- Lượt 2: chèn G → `GGG` → xóa → trống. Tổng 2 bóng.

Tính tay ví dụ 1: `board = "WRRBBW"`, `hand = "RB"`. Có hai bóng, mỗi bóng chỉ hữu ích khi chèn cạnh cặp cùng màu:
- Chèn R vào cặp RR → `WBBW`; chèn B vào cặp BB → `WW`. Hết bóng mà còn `WW` → thua.
- Đổi thứ tự (B trước rồi R) cũng ra `WW`. Chèn ở chỗ khác chỉ làm bàn dài thêm.
→ `-1`.

**Trực giác chung:** mỗi lượt có tối đa `(len(board)+1) × (số màu trong tay)` lựa chọn, và tay có ≤ 5 bóng → độ sâu ≤ 5. Cây trò chơi nhỏ, nhưng không quá nhỏ: `17 × 5 × 17 × 4 × …` có thể lên hàng triệu nhánh. Vì vậy cần **memo** (gộp trạng thái trùng) và **cắt tỉa** (bỏ những lựa chọn vô ích).

Một trường hợp **bẫy** quan trọng: `board = "RRWWRRBBRR"`, `hand = "WB"`. Đáp án là **2**:
- Chèn W **vào giữa cặp RR**: `RRWWRRBBR[W]R` → hai R bị tách, chẳng xóa gì.
- Chèn B: `RRWWRRBB[B]RWR` → xóa BBB → `RRWWRRRWR` → xóa RRR → `RRWWWR` → xóa WWW → `RRR` → xóa → trống!

Bài học: đôi khi chèn bóng **khác màu vào giữa một cặp cùng màu** cũng có ích. Quy tắc cắt tỉa phải giữ lại trường hợp này.

In [7]:
s = "RRWWRRBBRR"
s1 = s[:9] + "W" + s[9:]
print("chèn W tách cặp RR cuối:", s1, "->", clean(s1))
s2 = s1[:8] + "B" + s1[8:]
print("chèn B vào BB:", s2, "->", repr(clean(s2)))

chèn W tách cặp RR cuối: RRWWRRBBRWR -> RRWWRRBBRWR
chèn B vào BB: RRWWRRBBBRWR -> ''


## 5. Cách 1 — Brute Force: DFS thử mọi vị trí, mọi màu (có memo)

**Ý tưởng.** `solve(board, hand)` = số bóng ít nhất để dọn `board` với `hand` (chuẩn hóa sắp xếp), hoặc vô cực nếu không thể.
- Nếu `board` rỗng → 0.
- Nếu `hand` rỗng → vô cực.
- Ngược lại thử **mọi** vị trí `i` và **mọi** màu khác nhau `c` trong tay: `1 + solve(clean(board[:i]+c+board[i:]), hand bỏ c)`; lấy min.

Memo bằng `lru_cache` vì nhiều thứ tự chèn dẫn đến cùng trạng thái. Không cắt tỉa gì cả → đây là chuẩn đối chiếu đáng tin.

In [8]:
from functools import lru_cache
INF = float("inf")

def zuma_brute(board, hand):
    @lru_cache(maxsize=None)
    def solve(b, h):
        if not b:
            return 0
        if not h:
            return INF
        best = INF
        for j in range(len(h)):
            if j > 0 and h[j] == h[j - 1]:
                continue
            rest = h[:j] + h[j + 1:]
            for i in range(len(b) + 1):
                nb = clean(b[:i] + h[j] + b[i:])
                best = min(best, 1 + solve(nb, rest))
        return best
    ans = solve(board, "".join(sorted(hand)))
    return -1 if ans == INF else ans

print(zuma_brute("WRRBBW", "RB"), zuma_brute("WWRRBBWW", "WRBRW"), zuma_brute("G", "GGGGG"), zuma_brute("RRWWRRBBRR", "WB"))
assert zuma_brute("WRRBBW", "RB") == -1
assert zuma_brute("WWRRBBWW", "WRBRW") == 2
assert zuma_brute("G", "GGGGG") == 2
assert zuma_brute("RRWWRRBBRR", "WB") == 2

-1 2 2 2


**Độ phức tạp.** Số trạng thái khó đếm chính xác; thô sơ: mỗi tầng `(L+1)·5` lựa chọn, 5 tầng → `(17·5)·(18·4)·(19·3)·(20·2)·(21·1) ≈ 3·10^8` nhánh nếu không memo. Memo gộp bớt nhưng vẫn có thể rất nhiều trạng thái với board dài.

**Nút thắt.** Phần lớn lựa chọn vô ích: chèn một bóng `R` giữa `G` và `Y` chẳng bao giờ giúp được gì *trực tiếp*. Ta cần quy tắc để bỏ chúng.

In [9]:
import time
t = time.perf_counter()
print(zuma_brute("RRWWRRBBRRYYGGBB", "WBYGR"))
print(f"brute với board dài 16, tay 5 bóng: {time.perf_counter() - t:.2f}s")

4
brute với board dài 16, tay 5 bóng: 18.65s


## 6. Bước đột phá

Ta cần các quy tắc **cắt tỉa an toàn** (không làm mất đáp án tối ưu).

**Quy tắc 1 — Trùng màu thì chỉ thử một màu một lần.** Tay `"RRB"` chỉ có 2 lựa chọn màu: R và B. (Đã dùng ở brute force.)

**Quy tắc 2 — Chèn cạnh bóng cùng màu: chỉ cần thử một phía.** Chèn `R` vào **trước** hay **sau** một nhóm `RR` cho kết quả giống hệt. Vậy **nếu** `board[i-1] == c` **thì** bỏ qua vị trí `i` (vị trí bên trái nhóm đã được thử rồi).

**Quy tắc 3 — Chỉ chèn ở vị trí "có ích".** Chèn `c` ở vị trí `i` chỉ có ý nghĩa nếu:
- (a) `board[i] == c`: bóng mới **nối dài** một nhóm cùng màu; **hoặc**
- (b) `board[i-1] == board[i] != c`: bóng mới **tách đôi** một cặp/nhóm cùng màu (trường hợp bẫy `"RRWWRRBBRR"`).

Mọi vị trí khác (chèn giữa hai bóng khác màu nhau và khác `c`) bị bỏ.

**Vì sao hợp lý?** Một bóng chèn vào chỗ "lạc lõng" chỉ có thể có ích về sau khi có thêm bóng cùng màu chèn cạnh nó — nhưng khi đó ta có thể đổi thứ tự/vị trí để bóng ấy được chèn cạnh một bóng cùng màu, tức rơi vào (a). Đây là lập luận trực giác; LeetCode chấp nhận bộ quy tắc này (đây cũng là cắt tỉa trong lời giải chính thức). Trong notebook, ta **kiểm chứng bằng máy** trên hàng nghìn bộ test ngẫu nhiên so với brute force (mục 12) — đó là thái độ đúng khi dùng một quy tắc cắt tỉa mà chứng minh hình thức không đơn giản.

Tóm tắt thành mã giả:

```text
for i in 0..len(board):
    for c in các màu khác nhau trong hand:
        if i > 0 and board[i-1] == c: bỏ            # quy tắc 2
        ok = (i < len and board[i] == c) or (0 < i < len and board[i-1] == board[i] != c)   # quy tắc 3
        if ok: thử chèn
```

In [10]:
def useful_moves(board, hand):
    moves = []
    h = "".join(sorted(hand))
    for i in range(len(board) + 1):
        for j in range(len(h)):
            c = h[j]
            if j > 0 and h[j] == h[j - 1]:
                continue
            if i > 0 and board[i - 1] == c:
                continue
            ok = (i < len(board) and board[i] == c) or \
                 (0 < i < len(board) and board[i - 1] == board[i] != c)
            if ok:
                moves.append((i, c))
    return moves

b, h = "RRWWRRBBRR", "WB"
print("Tổng số nước nếu không cắt tỉa:", (len(b) + 1) * len(set(h)))
print("Số nước có ích:", len(useful_moves(b, h)), useful_moves(b, h))

Tổng số nước nếu không cắt tỉa: 22
Số nước có ích: 10 [(1, 'B'), (1, 'W'), (2, 'W'), (3, 'B'), (5, 'B'), (5, 'W'), (6, 'B'), (7, 'W'), (9, 'B'), (9, 'W')]


## 7. Các cách tiếp theo

### 7.1. Cách 2 — BFS không cắt tỉa (tìm số bước ít nhất theo lớp)

BFS tự nhiên phù hợp vì **mỗi bước tốn đúng 1 bóng** → số bước = số bóng. Trạng thái gặp đầu tiên có `board` rỗng là tối ưu. Dùng `visited` theo cặp `(board, hand đã sắp xếp)`.

In [11]:
from collections import deque

def zuma_bfs_plain(board, hand):
    start = (board, "".join(sorted(hand)))
    q = deque([(board, start[1], 0)])
    visited = {start}
    while q:
        b, h, step = q.popleft()
        for j in range(len(h)):
            if j > 0 and h[j] == h[j - 1]:
                continue
            rest = h[:j] + h[j + 1:]
            for i in range(len(b) + 1):
                nb = clean(b[:i] + h[j] + b[i:])
                if not nb:
                    return step + 1
                if rest and (nb, rest) not in visited:
                    visited.add((nb, rest))
                    q.append((nb, rest, step + 1))
    return -1

for b, h in [("WRRBBW", "RB"), ("WWRRBBWW", "WRBRW"), ("G", "GGGGG"), ("RRWWRRBBRR", "WB")]:
    print(b, h, "->", zuma_bfs_plain(b, h))

WRRBBW RB -> -1
WWRRBBWW WRBRW -> 2
G GGGGG -> 2
RRWWRRBBRR WB -> 2


**Code nộp LeetCode (Cách 2):** đúng nhưng có thể chậm với input khó; chỉ nên dùng để hiểu BFS.

In [12]:
class SolutionBFSPlain:
    def findMinStep(self, board: str, hand: str) -> int:
        return zuma_bfs_plain(board, hand)

print(SolutionBFSPlain().findMinStep("WWRRBBWW", "WRBRW"))

2


**Độ phức tạp Cách 2:** số trạng thái tối đa bằng số `(board, hand)` sinh ra được — trường hợp xấu vài trăm nghìn tới vài triệu; mỗi trạng thái sinh `O(L·|hand|)` con, mỗi con tốn `O(L)` để `clean`. Space bằng số trạng thái trong `visited`.

### 7.2. Cách 3 — BFS + cắt tỉa (lời giải chính)

Giống Cách 2 nhưng chỉ sinh các nước "có ích" theo quy tắc 2 và 3 ở mục 6. Số nhánh giảm mạnh.

In [13]:
def zuma_bfs_pruned(board, hand):
    h0 = "".join(sorted(hand))
    q = deque([(board, h0, 0)])
    visited = {(board, h0)}
    while q:
        b, h, step = q.popleft()
        for i in range(len(b) + 1):
            for j in range(len(h)):
                c = h[j]
                if j > 0 and h[j] == h[j - 1]:
                    continue                                   # quy tắc 1
                if i > 0 and b[i - 1] == c:
                    continue                                   # quy tắc 2
                ok = (i < len(b) and b[i] == c) or (0 < i < len(b) and b[i - 1] == b[i] != c)
                if not ok:
                    continue                                   # quy tắc 3
                nb = clean(b[:i] + c + b[i:])
                if not nb:
                    return step + 1
                rest = h[:j] + h[j + 1:]
                if rest and (nb, rest) not in visited:
                    visited.add((nb, rest))
                    q.append((nb, rest, step + 1))
    return -1

for b, h in [("WRRBBW", "RB"), ("WWRRBBWW", "WRBRW"), ("G", "GGGGG"), ("RRWWRRBBRR", "WB")]:
    print(b, h, "->", zuma_bfs_pruned(b, h))
assert zuma_bfs_pruned("RRWWRRBBRR", "WB") == 2

WRRBBW RB -> -1
WWRRBBWW WRBRW -> 2
G GGGGG -> 2
RRWWRRBBRR WB -> 2


**Code nộp LeetCode (Cách 3):**

In [14]:
from collections import deque

class SolutionBFS:
    def findMinStep(self, board: str, hand: str) -> int:
        def clean(s):
            st = []
            for c in s:
                if st and st[-1][0] != c and st[-1][1] >= 3:
                    st.pop()
                if st and st[-1][0] == c:
                    st[-1][1] += 1
                else:
                    st.append([c, 1])
            if st and st[-1][1] >= 3:
                st.pop()
            return "".join(c * n for c, n in st)

        h0 = "".join(sorted(hand))
        q = deque([(board, h0, 0)])
        seen = {(board, h0)}
        while q:
            b, h, step = q.popleft()
            for i in range(len(b) + 1):
                for j in range(len(h)):
                    c = h[j]
                    if (j and h[j] == h[j - 1]) or (i and b[i - 1] == c):
                        continue
                    if not ((i < len(b) and b[i] == c) or (0 < i < len(b) and b[i - 1] == b[i] != c)):
                        continue
                    nb = clean(b[:i] + c + b[i:])
                    if not nb:
                        return step + 1
                    rest = h[:j] + h[j + 1:]
                    if rest and (nb, rest) not in seen:
                        seen.add((nb, rest))
                        q.append((nb, rest, step + 1))
        return -1

print(SolutionBFS().findMinStep("RRWWRRBBRR", "WB"))

2


**Độ phức tạp Cách 3:** vẫn là hàm mũ theo `|hand|` trong trường hợp xấu nhất, nhưng mỗi tầng chỉ còn vài chục nhánh. Với ràng buộc `|board| <= 16`, `|hand| <= 5` chạy rất nhanh.

### 7.3. Cách 4 — DFS + memo + cắt tỉa

Cùng các quy tắc, nhưng viết đệ quy `solve(b, h)` có `lru_cache`. Ưu điểm: code ngắn, dễ thêm điều kiện; nhược điểm: phải đi hết cây con (không dừng sớm như BFS khi gặp đích ở tầng nông).

In [15]:
def zuma_dfs_pruned(board, hand):
    @lru_cache(maxsize=None)
    def solve(b, h):
        if not b:
            return 0
        if not h:
            return INF
        best = INF
        for i in range(len(b) + 1):
            for j in range(len(h)):
                c = h[j]
                if (j and h[j] == h[j - 1]) or (i and b[i - 1] == c):
                    continue
                if not ((i < len(b) and b[i] == c) or (0 < i < len(b) and b[i - 1] == b[i] != c)):
                    continue
                best = min(best, 1 + solve(clean(b[:i] + c + b[i:]), h[:j] + h[j + 1:]))
        return best
    ans = solve(board, "".join(sorted(hand)))
    return -1 if ans == INF else ans

for b, h in [("WRRBBW", "RB"), ("WWRRBBWW", "WRBRW"), ("G", "GGGGG"), ("RRWWRRBBRR", "WB")]:
    print(b, h, "->", zuma_dfs_pruned(b, h))

WRRBBW RB -> -1
WWRRBBWW WRBRW -> 2
G GGGGG -> 2
RRWWRRBBRR WB -> 2


**Code nộp LeetCode (Cách 4):**

In [16]:
from functools import lru_cache

class SolutionDFS:
    def findMinStep(self, board: str, hand: str) -> int:
        return zuma_dfs_pruned(board, hand)

print(SolutionDFS().findMinStep("WWRRBBWW", "WRBRW"))

2


**Độ phức tạp Cách 4:** tương tự Cách 3 về số trạng thái; mỗi trạng thái được tính đúng một lần nhờ memo.

## 8. TRACE từng bước

Trace **Cách 3 (BFS + cắt tỉa)** trên ví dụ 2: `board = "WWRRBBWW"`, `hand = "WRBRW"` (chuẩn hóa `"BRRWW"`).

Các cột: `lớp` (số bóng đã dùng), trạng thái lấy ra, nước đi `(vị trí, màu)`, bàn sau khi chèn và sau khi `clean`.

| lớp | board | hand | nước | sau clean |
|---|---|---|---|---|
| 0 | WWRRBBWW | BRRWW | (4,B) | WWRRWW |
| 0 | WWRRBBWW | BRRWW | (2,R) | WWBBWW |
| 1 | WWBBWW | BRWW | (2,B) | **rỗng** → trả về 2 |

In [17]:
def trace_bfs(board, hand, limit=25):
    h0 = "".join(sorted(hand))
    q = deque([(board, h0, 0)])
    visited = {(board, h0)}
    printed = 0
    while q:
        b, h, step = q.popleft()
        print(f"[lớp {step}] lấy ra board={b!r} hand={h!r}")
        for i in range(len(b) + 1):
            for j in range(len(h)):
                c = h[j]
                if (j and h[j] == h[j - 1]) or (i and b[i - 1] == c):
                    continue
                if not ((i < len(b) and b[i] == c) or (0 < i < len(b) and b[i - 1] == b[i] != c)):
                    continue
                ins = b[:i] + c + b[i:]
                nb = clean(ins)
                rest = h[:j] + h[j + 1:]
                printed += 1
                if printed <= limit:
                    print(f"     chèn {c} tại {i}: {ins!r} -> clean -> {nb!r}  (còn {rest!r})")
                if not nb:
                    print(f"  => bàn trống sau {step + 1} bóng")
                    return step + 1
                if rest and (nb, rest) not in visited:
                    visited.add((nb, rest))
                    q.append((nb, rest, step + 1))
    print("  => không thể dọn sạch")
    return -1

trace_bfs("WWRRBBWW", "WRBRW")

[lớp 0] lấy ra board='WWRRBBWW' hand='BRRWW'
     chèn W tại 0: 'WWWRRBBWW' -> clean -> 'RRBBWW'  (còn 'BRRW')
     chèn B tại 1: 'WBWRRBBWW' -> clean -> 'WBWRRBBWW'  (còn 'RRWW')
     chèn R tại 1: 'WRWRRBBWW' -> clean -> 'WRWRRBBWW'  (còn 'BRWW')
     chèn R tại 2: 'WWRRRBBWW' -> clean -> 'WWBBWW'  (còn 'BRWW')
     chèn B tại 3: 'WWRBRBBWW' -> clean -> 'WWRBRBBWW'  (còn 'RRWW')
     chèn W tại 3: 'WWRWRBBWW' -> clean -> 'WWRWRBBWW'  (còn 'BRRW')
     chèn B tại 4: 'WWRRBBBWW' -> clean -> 'WWRRWW'  (còn 'RRWW')
     chèn R tại 5: 'WWRRBRBWW' -> clean -> 'WWRRBRBWW'  (còn 'BRWW')
     chèn W tại 5: 'WWRRBWBWW' -> clean -> 'WWRRBWBWW'  (còn 'BRRW')
     chèn W tại 6: 'WWRRBBWWW' -> clean -> 'WWRRBB'  (còn 'BRRW')
     chèn B tại 7: 'WWRRBBWBW' -> clean -> 'WWRRBBWBW'  (còn 'RRWW')
     chèn R tại 7: 'WWRRBBWRW' -> clean -> 'WWRRBBWRW'  (còn 'BRWW')
[lớp 1] lấy ra board='RRBBWW' hand='BRRW'
     chèn R tại 0: 'RRRBBWW' -> clean -> 'BBWW'  (còn 'BRW')
     chèn B tại 1: 'RBRBBWW' -> clea

2

Trace trường hợp bẫy để thấy nước "tách cặp" được sinh ra:

In [18]:
trace_bfs("RRWWRRBBRR", "WB")

[lớp 0] lấy ra board='RRWWRRBBRR' hand='BW'
     chèn B tại 1: 'RBRWWRRBBRR' -> clean -> 'RBRWWRRBBRR'  (còn 'W')
     chèn W tại 1: 'RWRWWRRBBRR' -> clean -> 'RWRWWRRBBRR'  (còn 'B')
     chèn W tại 2: 'RRWWWRRBBRR' -> clean -> 'BBRR'  (còn 'B')
     chèn B tại 3: 'RRWBWRRBBRR' -> clean -> 'RRWBWRRBBRR'  (còn 'W')
     chèn B tại 5: 'RRWWRBRBBRR' -> clean -> 'RRWWRBRBBRR'  (còn 'W')
     chèn W tại 5: 'RRWWRWRBBRR' -> clean -> 'RRWWRWRBBRR'  (còn 'B')
     chèn B tại 6: 'RRWWRRBBBRR' -> clean -> 'RRWW'  (còn 'W')
     chèn W tại 7: 'RRWWRRBWBRR' -> clean -> 'RRWWRRBWBRR'  (còn 'B')
     chèn B tại 9: 'RRWWRRBBRBR' -> clean -> 'RRWWRRBBRBR'  (còn 'W')
     chèn W tại 9: 'RRWWRRBBRWR' -> clean -> 'RRWWRRBBRWR'  (còn 'B')
[lớp 1] lấy ra board='RBRWWRRBBRR' hand='W'
     chèn W tại 3: 'RBRWWWRRBBRR' -> clean -> ''  (còn '')
  => bàn trống sau 2 bóng


2

Và trace hàm `clean` để thấy stack thay đổi thế nào trong phản ứng dây chuyền:

In [19]:
def trace_clean(s):
    stack = []
    for c in s:
        if stack and stack[-1][0] != c and stack[-1][1] >= 3:
            print(f"  gặp {c}: xóa nhóm {stack[-1][0]}x{stack[-1][1]}")
            stack.pop()
        if stack and stack[-1][0] == c:
            stack[-1][1] += 1
        else:
            stack.append([c, 1])
        print(f"đọc {c}: stack = {[(x, n) for x, n in stack]}")
    if stack and stack[-1][1] >= 3:
        print(f"  hết chuỗi: xóa nhóm {stack[-1][0]}x{stack[-1][1]}")
        stack.pop()
    res = "".join(c * n for c, n in stack)
    print("kết quả:", repr(res))
    return res

trace_clean("RRWWRRBBBRWR")

đọc R: stack = [('R', 1)]
đọc R: stack = [('R', 2)]
đọc W: stack = [('R', 2), ('W', 1)]
đọc W: stack = [('R', 2), ('W', 2)]
đọc R: stack = [('R', 2), ('W', 2), ('R', 1)]
đọc R: stack = [('R', 2), ('W', 2), ('R', 2)]
đọc B: stack = [('R', 2), ('W', 2), ('R', 2), ('B', 1)]
đọc B: stack = [('R', 2), ('W', 2), ('R', 2), ('B', 2)]
đọc B: stack = [('R', 2), ('W', 2), ('R', 2), ('B', 3)]
  gặp R: xóa nhóm Bx3
đọc R: stack = [('R', 2), ('W', 2), ('R', 3)]
  gặp W: xóa nhóm Rx3
đọc W: stack = [('R', 2), ('W', 3)]
  gặp R: xóa nhóm Wx3
đọc R: stack = [('R', 3)]
  hết chuỗi: xóa nhóm Rx3
kết quả: ''


''

## 9. Code nộp LeetCode

In [20]:
from collections import deque

class Solution:
    def findMinStep(self, board: str, hand: str) -> int:
        def clean(s: str) -> str:
            st = []                                  # [màu, số lượng]
            for c in s:
                if st and st[-1][0] != c and st[-1][1] >= 3:
                    st.pop()
                if st and st[-1][0] == c:
                    st[-1][1] += 1
                else:
                    st.append([c, 1])
            if st and st[-1][1] >= 3:
                st.pop()
            return "".join(c * n for c, n in st)

        start_hand = "".join(sorted(hand))
        queue = deque([(board, start_hand, 0)])
        visited = {(board, start_hand)}
        while queue:
            b, h, step = queue.popleft()
            for i in range(len(b) + 1):
                for j, c in enumerate(h):
                    if j > 0 and h[j - 1] == c:              # màu trùng trong tay
                        continue
                    if i > 0 and b[i - 1] == c:              # chèn cạnh cùng màu: chỉ thử phía trái
                        continue
                    extend = i < len(b) and b[i] == c
                    split = 0 < i < len(b) and b[i - 1] == b[i] != c
                    if not (extend or split):
                        continue
                    nb = clean(b[:i] + c + b[i:])
                    if not nb:
                        return step + 1
                    rest = h[:j] + h[j + 1:]
                    if rest and (nb, rest) not in visited:
                        visited.add((nb, rest))
                        queue.append((nb, rest, step + 1))
        return -1

sol = Solution()
assert sol.findMinStep("WRRBBW", "RB") == -1
assert sol.findMinStep("WWRRBBWW", "WRBRW") == 2
assert sol.findMinStep("G", "GGGGG") == 2
assert sol.findMinStep("RRWWRRBBRR", "WB") == 2
print("Solution OK")

Solution OK


## 10. Vì sao thuật toán đúng

**Hàm `clean` đúng.** Bất biến của stack: nó mô tả phần đã đọc sau khi đã xóa mọi nhóm ≥ 3 **đã kết thúc**; chỉ có nhóm ở **đỉnh** có thể đang ≥ 3 (vì còn có thể kéo dài). Khi đọc bóng khác màu, nhóm đỉnh kết thúc → nếu ≥ 3 thì xóa; nhóm bên dưới lộ ra và nếu cùng màu với bóng mới thì được nối dài — đó chính là phản ứng dây chuyền. Kết thúc chuỗi xử lý nhóm đỉnh lần cuối. Mọi nhóm bên dưới đỉnh có kích thước < 3 (vì nếu ≥ 3 thì đã bị xóa khi kết thúc), nên kết quả không còn nhóm nào ≥ 3.

**BFS cho số bước ít nhất.** Mỗi cạnh trong đồ thị trạng thái tương ứng đúng 1 bóng. BFS duyệt theo lớp tăng dần, nên lần đầu tạo ra bàn rỗng là ở lớp nhỏ nhất. `visited` không làm mất đáp án vì trạng thái `(board, hand)` quyết định hoàn toàn tương lai — gặp lại ở lớp lớn hơn không thể tốt hơn.

**Cắt tỉa.**
- Quy tắc 1, 2 hiển nhiên an toàn (các nước bị bỏ cho **cùng** trạng thái kết quả với một nước đã thử).
- Quy tắc 3 là heuristic được cộng đồng và lời giải chính thức dùng; ý chính: một bóng chèn ở chỗ "không kề cùng màu và không tách cặp" thì không gây xóa ngay, và có thể hoán đổi vị trí/thời điểm chèn để thu được cùng kết quả với nước "có ích". Notebook kiểm chứng điều này bằng **hàng nghìn test ngẫu nhiên** so với brute force không cắt tỉa (mục 12). Nếu bạn dùng ở phỏng vấn, hãy nêu rõ đây là cắt tỉa đã kiểm chứng, và brute force có memo là phương án an toàn.

## 11. Phân tích Time/Space

Gọi `L = |board| <= 16`, `H = |hand| <= 5`.

- Mỗi trạng thái sinh tối đa `(L + H + 1) · H` nước (bàn có thể dài thêm tối đa `H`), mỗi nước `clean` tốn `O(L + H)`.
- Số trạng thái: tối đa cỡ `∏ (L+t)·(H-t)` theo từng tầng nếu không memo; memo và cắt tỉa giảm mạnh. Trong thực tế (ràng buộc LeetCode) chỉ vài nghìn trạng thái.
- Space: `O(số trạng thái · (L + H))` cho `visited`/memo.

So sánh số trạng thái mà mỗi cách thăm:

In [21]:
def count_states_bfs(board, hand, pruned):
    h0 = "".join(sorted(hand))
    q = deque([(board, h0)])
    visited = {(board, h0)}
    while q:
        b, h = q.popleft()
        for i in range(len(b) + 1):
            for j in range(len(h)):
                c = h[j]
                if j and h[j] == h[j - 1]:
                    continue
                if pruned:
                    if i and b[i - 1] == c:
                        continue
                    if not ((i < len(b) and b[i] == c) or (0 < i < len(b) and b[i - 1] == b[i] != c)):
                        continue
                nb = clean(b[:i] + c + b[i:])
                rest = h[:j] + h[j + 1:]
                if nb and rest and (nb, rest) not in visited:
                    visited.add((nb, rest))
                    q.append((nb, rest))
    return len(visited)

for b, h in [("WWRRBBWW", "WRBRW"), ("RRWWRRBBRR", "WB"), ("RBYYBBRRB", "YRBGB")]:
    print(f"{b:<12} {h:<6} không cắt tỉa: {count_states_bfs(b, h, False):>6} trạng thái | cắt tỉa: {count_states_bfs(b, h, True):>5}")

WWRRBBWW     WRBRW  không cắt tỉa:   3577 trạng thái | cắt tỉa:   783
RRWWRRBBRR   WB     không cắt tỉa:     19 trạng thái | cắt tỉa:    11


RBYYBBRRB    YRBGB  không cắt tỉa:  13839 trạng thái | cắt tỉa:  1392


## 12. Bộ test

In [22]:
methods = {
    "brute": zuma_brute,
    "bfs_plain": zuma_bfs_plain,
    "bfs_pruned": zuma_bfs_pruned,
    "dfs_pruned": zuma_dfs_pruned,
    "SolutionBFS": SolutionBFS().findMinStep,
    "Solution": Solution().findMinStep,
}
cases = [
    ("WRRBBW", "RB", -1),
    ("WWRRBBWW", "WRBRW", 2),
    ("G", "GGGGG", 2),
    ("RRWWRRBBRR", "WB", 2),          # bẫy: phải tách cặp
    ("RBYYBBRRB", "YRBGB", 3),
    ("R", "B", -1),                   # màu không có
    ("RR", "R", 1),
    ("RGBYW", "RRGGB", -1),
    ("WWBBWBBWW", "BB", -1),
]
for b, h, exp in cases:
    for name, f in methods.items():
        got = f(b, h)
        assert got == exp, (name, b, h, got, exp)
print("Ví dụ + edge case: OK")

Ví dụ + edge case: OK


In [23]:
import random

def random_board(n, colors):
    # sinh board không có 3 bóng liên tiếp cùng màu
    s = ""
    while len(s) < n:
        c = random.choice(colors)
        if len(s) >= 2 and s[-1] == s[-2] == c:
            continue
        s += c
    return s

random.seed(488)
checked = 0
for trial in range(1500):
    colors = random.choice(["RG", "RGB", "RGBY"])
    b = random_board(random.randint(1, 7), colors)
    h = "".join(random.choice(colors) for _ in range(random.randint(1, 4)))
    exp = zuma_brute(b, h)
    for name, f in methods.items():
        if name == "brute":
            continue
        got = f(b, h)
        assert got == exp, (name, b, h, got, exp)
    checked += 1
print(f"{checked} test ngẫu nhiên (seed 488): các cách khớp brute force")

1500 test ngẫu nhiên (seed 488): các cách khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Chỉ xóa đúng 3 bóng / không xử lý dây chuyền

**Code sai:** `s.replace("RRR", "")` hoặc chỉ xóa một lần.

**Vì sao sai:** nhóm có thể dài 4 (`RRRR`) và sau khi xóa có thể sinh nhóm mới (`WWBBBWW` → `WWWW` → rỗng).

In [24]:
def clean_wrong(s):
    for c in "RYBGW":
        s = s.replace(c * 3, "")
    return s

for s in ["WWBBBWW", "RRRRG"]:
    print(f"{s}: sai -> {clean_wrong(s)!r}, đúng -> {clean(s)!r}")

WWBBBWW: sai -> 'W', đúng -> ''
RRRRG: sai -> 'RG', đúng -> 'G'


**Code đúng:** dùng stack như `clean`.

### Lỗi 2 — Cắt tỉa quá tay: chỉ chèn cạnh bóng cùng màu

**Code sai:** chỉ thử các vị trí có `board[i] == c`.

**Vì sao sai:** bỏ mất nước "tách cặp" → `"RRWWRRBBRR", "WB"` trả về `-1` thay vì `2`.

In [25]:
def zuma_overpruned(board, hand):
    h0 = "".join(sorted(hand))
    q = deque([(board, h0, 0)]); vis = {(board, h0)}
    while q:
        b, h, step = q.popleft()
        for i in range(len(b) + 1):
            for j in range(len(h)):
                c = h[j]
                if not (i < len(b) and b[i] == c):     # chỉ nối dài nhóm
                    continue
                nb = clean(b[:i] + c + b[i:])
                if not nb:
                    return step + 1
                rest = h[:j] + h[j + 1:]
                if rest and (nb, rest) not in vis:
                    vis.add((nb, rest)); q.append((nb, rest, step + 1))
    return -1

print("cắt tỉa quá tay:", zuma_overpruned("RRWWRRBBRR", "WB"), "| đúng:", Solution().findMinStep("RRWWRRBBRR", "WB"))

cắt tỉa quá tay: -1 | đúng: 2


**Code đúng:** giữ điều kiện `split = 0 < i < len(b) and b[i-1] == b[i] != c`.

### Lỗi 3 — Không chuẩn hóa `hand` → memo/visited kém hiệu quả

**Code sai:** dùng `hand` nguyên thứ tự làm khóa. `"RB"` và `"BR"` bị coi là hai trạng thái khác nhau → số trạng thái nhân lên tới `H!` lần.

**Code đúng:** `"".join(sorted(hand))` một lần ở đầu; khi bỏ một bóng bằng slicing thì chuỗi vẫn sắp xếp.

### Lỗi 4 — Trả về khi lấy ra khỏi hàng đợi thay vì khi sinh ra / quên `rest` rỗng

- **Code sai:** đưa mọi trạng thái vào hàng đợi rồi chỉ kiểm tra `if not b: return step` khi **lấy ra** — vẫn đúng, nhưng phải duyệt hết cả lớp hiện tại trước khi tới lớp chứa đích, và hàng đợi phình to.
- **Code sai hơn:** kiểm tra lúc lấy ra nhưng trả `step + 1` → lệch 1.
- **Code đúng:** kiểm tra `if not nb: return step + 1` ngay khi **sinh ra** trạng thái con; trạng thái có `rest` rỗng mà bàn chưa trống là ngõ cụt → không đưa vào hàng đợi.

Lỗi 3 minh họa bằng số liệu — số trạng thái khi không chuẩn hóa `hand`:

In [26]:
# So sánh số trạng thái khi không chuẩn hóa hand
def count_states_unsorted(board, hand):
    q = deque([(board, hand)]); vis = {(board, hand)}
    while q:
        b, h = q.popleft()
        for i in range(len(b) + 1):
            for j in range(len(h)):
                nb = clean(b[:i] + h[j] + b[i:]); rest = h[:j] + h[j + 1:]
                if nb and rest and (nb, rest) not in vis:
                    vis.add((nb, rest)); q.append((nb, rest))
    return len(vis)

print("không sắp xếp hand:", count_states_unsorted("WWRRBBWW", "WRBRW"),
      "| sắp xếp:", count_states_bfs("WWRRBBWW", "WRBRW", False))

không sắp xếp hand: 4252 | sắp xếp: 3577


## 14. Mẹo & nhận dạng pattern

- Ràng buộc **rất nhỏ** (`<= 16`, `<= 5`) + "số bước ít nhất" → **BFS trên không gian trạng thái** hoặc **DFS + memo**.
- Trạng thái gồm chuỗi + multiset → **chuẩn hóa multiset** (sắp xếp / đếm) để memo hiệu quả.
- "Xóa nhóm liên tiếp ≥ k, dây chuyền" → **stack (màu, đếm)** (LC 1209 Remove All Adjacent Duplicates in String II).
- Cắt tỉa: tìm những nước **cho cùng kết quả** (đối xứng) và những nước **chắc chắn vô ích**; luôn kiểm chứng với brute force.
- BFS dừng ngay khi **sinh ra** đích nếu mọi cạnh có cùng trọng số.

## 15. Biến thể

1. **Hàng bóng vòng tròn** (đầu nối cuối): `clean` phải xử lý nhóm vắt qua đầu/cuối.
2. **Xóa khi ≥ k bóng** (thay vì 3): đổi hằng số trong `clean`.
3. **Tối đa số bóng bị xóa** với số lượt cố định — chuyển từ BFS sang DFS + memo tối ưu giá trị.
4. **Remove Boxes (LC 546):** cùng "họ" xóa nhóm liên tiếp nhưng tối đa điểm, DP 3 chiều.
5. **Candy Crush (LC 723):** mô phỏng xóa nhóm ≥ 3 trên lưới 2D.

Demo biến thể 2 — `clean` với ngưỡng `k`:

In [27]:
def clean_k(s, k):
    st = []
    for c in s:
        if st and st[-1][0] != c and st[-1][1] >= k:
            st.pop()
        if st and st[-1][0] == c:
            st[-1][1] += 1
        else:
            st.append([c, 1])
    if st and st[-1][1] >= k:
        st.pop()
    return "".join(c * n for c, n in st)

print(clean_k("RRBBBRR", 3), "|", clean_k("RRBBBRR", 4), "|", clean_k("RRBBBBRR", 4))
assert clean_k("RRBBBBRR", 4) == ""      # BBBB bị xóa -> RRRR cũng đủ 4 -> xóa tiếp

 | RRBBBRR | 


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. DFS + memo, không cắt tỉa | Thử mọi vị trí, mọi màu | mũ, lớn nhất | số trạng thái | Chuẩn đối chiếu |
| 2. BFS không cắt tỉa | Duyệt theo lớp, dừng sớm | mũ | số trạng thái | Hiểu BFS, input nhỏ |
| 3. BFS + cắt tỉa | Chỉ chèn ở chỗ nối dài / tách cặp | mũ nhưng nhỏ | nhỏ | Lời giải nộp |
| 4. DFS + memo + cắt tỉa | Đệ quy, memo theo `(board, hand)` | tương tự 3 | tương tự | Thích viết đệ quy |

## 17. Bài tập

**Bài 1 (đoán output).** `clean("YYBBBYRR")` trả về gì?

**Bài 2 (tính tay).** `board = "RRGG"`, `hand = "RG"`. Đáp án là bao nhiêu? Liệt kê từng nước.

**Bài 3 (điền TODO).** Hoàn thiện hàm `useful_todo(b, i, c)` trả `True` nếu chèn `c` ở vị trí `i` là nước có ích theo quy tắc 2 và 3.

In [28]:
def useful_todo(b, i, c):
    # TODO: nếu i > 0 và b[i-1] == c -> False
    # TODO: trả True nếu (i < len(b) và b[i] == c) hoặc (0 < i < len(b) và b[i-1] == b[i] != c)
    raise NotImplementedError

# Không gọi khi chạy notebook. Tự kiểm tra:
# assert useful_todo("RRWW", 0, "R") and not useful_todo("RRWW", 1, "R") and useful_todo("RRWW", 1, "B")

**Bài 4 (tự viết lại).** Viết lại `Solution` bằng DFS + memo (không nhìn mục 7.3).

**Bài 5 (mini challenge).** Viết `max_removed(board, hand)`: dùng **tất cả hoặc một phần** bóng trong tay, tối đa hóa **số bóng trên bàn ban đầu** bị xóa (bóng chèn vào không tính). Không cần cắt tỉa; dùng DFS + memo.

In [29]:
def max_removed_todo(board, hand):
    # TODO
    raise NotImplementedError

# Không gọi khi chạy notebook.

## 18. Gợi ý 3 mức

**Bài 1–2**
- Mức 1: Chạy `trace_clean` bằng đầu.
- Mức 2: Bài 1: xóa BBB → `YYYRR` → xóa YYY.
- Mức 3: Bài 2: chèn R vào RR → `GG`, chèn G → trống. Đáp án 2.

**Bài 3**
- Mức 1: Hai điều kiện "nối dài" và "tách cặp".
- Mức 2: Quy tắc 2 kiểm tra **trước**.
- Mức 3: Chú ý `0 < i < len(b)` để không truy cập ngoài chuỗi.

**Bài 4**
- Mức 1: `@lru_cache` trên hàm nhận `(b, h)` là chuỗi.
- Mức 2: Cơ sở: `b` rỗng → 0; `h` rỗng → vô cực.
- Mức 3: Trả `-1` nếu kết quả là vô cực.

**Bài 5**
- Mức 1: Số bóng ban đầu bị xóa = số bóng đã chèn + `len(board) - len(board mới)`... cẩn thận!
- Mức 2: Đơn giản hơn: tính "tổng số bóng bị xóa" = `len(b) + 1 - len(clean(...))` rồi trừ số bóng chèn bị xóa — khó tách. Cách dễ: đánh dấu bóng ban đầu bằng chữ hoa, bóng chèn bằng chữ thường, `clean` so màu không phân biệt hoa thường.
- Mức 3: Kết quả = `len(board)` trừ số chữ hoa còn lại ở trạng thái tốt nhất (lấy max qua mọi trạng thái đạt được).

## 19. Đáp án tham khảo

In [30]:
# Bài 1, 2
assert clean("YYBBBYRR") == "RR"
assert Solution().findMinStep("RRGG", "RG") == 2

# Bài 3
def useful_sol(b, i, c):
    if i > 0 and b[i - 1] == c:
        return False
    return (i < len(b) and b[i] == c) or (0 < i < len(b) and b[i - 1] == b[i] != c)
assert useful_sol("RRWW", 0, "R") and not useful_sol("RRWW", 1, "R") and useful_sol("RRWW", 1, "B")
assert sorted((i, c) for i in range(5) for c in "BRW" if useful_sol("RRWW", i, c)) == \
       sorted(useful_moves("RRWW", "BRW"))

# Bài 4
class SolutionEx4:
    def findMinStep(self, board: str, hand: str) -> int:
        @lru_cache(maxsize=None)
        def go(b, h):
            if not b:
                return 0
            best = INF
            for i in range(len(b) + 1):
                for j, c in enumerate(h):
                    if j and h[j - 1] == c:
                        continue
                    if useful_sol(b, i, c):
                        best = min(best, 1 + go(clean(b[:i] + c + b[i:]), h[:j] + h[j + 1:]))
            return best
        r = go(board, "".join(sorted(hand)))
        return -1 if r == INF else r

for b, h, exp in cases:
    assert SolutionEx4().findMinStep(b, h) == exp
print("Bài 1–4 OK")

Bài 1–4 OK


In [31]:
# Bài 5: bóng ban đầu = chữ hoa, bóng chèn = chữ thường; clean so màu theo .upper()
def clean_mixed(s):
    st = []                                  # [màu viết hoa, danh sách ký tự]
    for ch in s:
        c = ch.upper()
        if st and st[-1][0] != c and len(st[-1][1]) >= 3:
            st.pop()
        if st and st[-1][0] == c:
            st[-1][1].append(ch)
        else:
            st.append([c, [ch]])
    if st and len(st[-1][1]) >= 3:
        st.pop()
    return "".join("".join(g) for _, g in st)

def max_removed(board, hand):
    best = 0
    @lru_cache(maxsize=None)
    def go(b, h):
        nonlocal best
        removed = len(board) - sum(ch.isupper() for ch in b)
        best = max(best, removed)
        for j in range(len(h)):
            if j and h[j] == h[j - 1]:
                continue
            for i in range(len(b) + 1):
                go(clean_mixed(b[:i] + h[j].lower() + b[i:]), h[:j] + h[j + 1:])
    go(board, "".join(sorted(hand)))
    return best

print(max_removed("WRRBBW", "RB"))       # xóa RR, BB, nhưng WW còn -> 4
assert max_removed("WRRBBW", "RB") == 4
assert max_removed("WWRRBBWW", "WRBRW") == 8
assert max_removed("RGB", "Y") == 0
print("Bài 5 OK")

4


Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template BFS trạng thái:**

```python
start = (board, "".join(sorted(hand)))
q = deque([(*start, 0)]); seen = {start}
while q:
    b, h, d = q.popleft()
    for i in range(len(b) + 1):
        for j, c in enumerate(h):
            if (j and h[j-1] == c) or (i and b[i-1] == c): continue
            if not ((i < len(b) and b[i] == c) or (0 < i < len(b) and b[i-1] == b[i] != c)): continue
            nb = clean(b[:i] + c + b[i:])
            if not nb: return d + 1
            rest = h[:j] + h[j+1:]
            if rest and (nb, rest) not in seen: seen.add((nb, rest)); q.append((nb, rest, d + 1))
return -1
```

**Checklist trước khi submit:**

- [ ] `clean` xử lý nhóm ≥ 3 (không chỉ đúng 3) và phản ứng dây chuyền.
- [ ] `hand` đã sắp xếp; bỏ qua màu trùng.
- [ ] Giữ nước **tách cặp** (`b[i-1] == b[i] != c`) — test `"RRWWRRBBRR", "WB"` → 2.
- [ ] Trả về `-1` khi không thể.
- [ ] Dừng ngay khi bàn rỗng, trả `step + 1`.
- [ ] Đã đối chiếu với brute force không cắt tỉa trên dữ liệu ngẫu nhiên nhỏ.